# 沐曦 GPU 运行 Hibou 说明文档

[Hibou](https://github.com/HistAI/hibou) 是面向数字病理图像的基础视觉 Transformer。Hibou 上游代码依据 [Apache License 2.0](https://github.com/HistAI/hibou/blob/main/LICENSE) 发布，Hugging Face [histai/hibou-L](https://huggingface.co/histai/hibou-L) 模型卡同样标注 Apache License 2.0；本教程未修改上游源代码或模型权重。您可通过上述链接查阅源代码、许可证全文、模型卡、版权与归属声明。

本教程在沐曦 GPU 上执行三个独立推理任务：`model(image)` 得到 CLS embedding、`forward_features(image)` 得到 patch tokens，以及 `forward_features(image, return_intermediate=True)` 得到 24 层中间表示。示例图使用上游仓库公开的 `images/sample.png`，运行路径由参数单元注入，不把图片、权重或结果文件复制到本目录。输出会保存在 Notebook 的执行结果中；可选张量快照写入外部临时目录。

Hibou-L 仅用于研究和工程复现，不代表临床图像、诊断结论、临床准确性或监管验证。

## 1. 安装

以下命令仅用于在 Notebook 外准备环境，不在 Notebook 中自动安装。项目目录只需保留源码 checkout、`tutorial.ipynb` 和上游许可证；模型权重保存到外部目录（例如 `/pde_hpc/szheng/hibou-L`）。

### 1.1 沐曦容器

```bash
docker run -it --name hibou-l-maca \
  --device=/dev/mxcd --device=/dev/dri --group-add video \
  --shm-size=8G --security-opt seccomp=unconfined \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -v /workspace:/workspace -v /pde_hpc/szheng:/pde_hpc/szheng \
  <MACA_PYTORCH_IMAGE> /bin/bash
```

### 1.2 上游源码、依赖和权重

```bash
cd /workspace
git clone https://github.com/HistAI/hibou.git
cd hibou
pip install -r requirements.txt
pip install safetensors
pip install -e .
huggingface-cli download histai/hibou-L \
  --local-dir /pde_hpc/szheng/hibou-L \
  --token "$HF_TOKEN"
```

请在终端外部设置 `HF_TOKEN`，不要把真实 token 写入 Notebook、日志或 Git。权重许可证和访问条件以 Hugging Face 模型页为准。

### 1.3 运行参数与模型加载

下面代码单元是共享参数锚点。`HIBOU_SOURCE` 指向上游源码 checkout，`HIBOU_IMAGE_PATH` 指向 checkout 内随仓库发布的 `images/sample.png`，`HIBOU_WEIGHTS_PATH` 指向外部 Hibou-L `model.safetensors`，`HIBOU_RESULT_DIR` 默认位于 `/tmp`。Hibou-L 配置为 1024 hidden size、24 层、16 heads、14×14 patch 和 4 register tokens；权重中的 query/key/value 会在加载边界处合并为上游 `qkv` 参数。

In [1]:
import gc
import json
import os
import sys
import time
import types
import warnings
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torchvision import transforms

warnings.filterwarnings("ignore", message=r"xFormers.*")
SOURCE_ROOT = Path(os.environ.get("HIBOU_SOURCE", "/workspace/hibou")).expanduser().resolve()
IMAGE_PATH = Path(os.environ.get("HIBOU_IMAGE_PATH", str(SOURCE_ROOT / "images" / "sample.png"))).expanduser().resolve()
WEIGHTS_PATH = Path(os.environ.get("HIBOU_WEIGHTS_PATH", "/pde_hpc/szheng/hibou-L/model.safetensors")).expanduser().resolve()
RESULT_DIR = Path(os.environ.get("HIBOU_RESULT_DIR", "/tmp/hibou-L-results")).expanduser().resolve()
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MODEL_ID = "histai/hibou-L"
SEED = 20240901
MODEL_CONFIG = dict(
    img_size=224, patch_size=14, arch="vit_large",
    layerscale=1.0, ffn_layer="swiglufused", block_chunks=0,
    num_register_tokens=4, interpolate_offset=0, interpolate_antialias=True,
)
NORMALIZE_MEAN = [0.7068, 0.5755, 0.7220]
NORMALIZE_STD = [0.1950, 0.2316, 0.1816]
if not torch.cuda.is_available():
    raise RuntimeError("未检测到可用的沐曦 GPU，请检查 MACA 容器设备映射。")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f"上游示例图不存在：{IMAGE_PATH}")
if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(f"Hibou-L 权重不存在：{WEIGHTS_PATH}")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

source_package = SOURCE_ROOT / "hibou"
for module_name in list(sys.modules):
    if module_name == "hibou" or module_name.startswith("hibou."):
        del sys.modules[module_name]
package = types.ModuleType("hibou")
package.__path__ = [str(source_package)]
package.__package__ = "hibou"
sys.modules["hibou"] = package
from hibou.models import build_model
from safetensors.torch import load_file

def convert_hf_state_dict(hf_state):
    converted = {
        "cls_token": hf_state["embeddings.cls_token"],
        "mask_token": hf_state["embeddings.mask_token"],
        "patch_embed.proj.weight": hf_state["embeddings.patch_embeddings.projection.weight"],
        "patch_embed.proj.bias": hf_state["embeddings.patch_embeddings.projection.bias"],
        "pos_embed": hf_state["embeddings.position_embeddings"],
        "register_tokens": hf_state["embeddings.register_tokens"],
        "norm.weight": hf_state["layernorm.weight"],
        "norm.bias": hf_state["layernorm.bias"],
    }
    for index in range(24):
        source = f"encoder.layer.{index}"
        target = f"blocks.{index}"
        for name in ("norm1.weight", "norm1.bias", "norm2.weight", "norm2.bias"):
            converted[f"{target}.{name}"] = hf_state[f"{source}.{name}"]
        qkv_names = ("query", "key", "value")
        converted[f"{target}.attn.qkv.weight"] = torch.cat([hf_state[f"{source}.attention.attention.{name}.weight"] for name in qkv_names], dim=0)
        converted[f"{target}.attn.qkv.bias"] = torch.cat([hf_state[f"{source}.attention.attention.{name}.bias"] for name in qkv_names], dim=0)
        converted[f"{target}.attn.proj.weight"] = hf_state[f"{source}.attention.output.dense.weight"]
        converted[f"{target}.attn.proj.bias"] = hf_state[f"{source}.attention.output.dense.bias"]
        converted[f"{target}.ls1.gamma"] = hf_state[f"{source}.layer_scale1.lambda1"]
        converted[f"{target}.ls2.gamma"] = hf_state[f"{source}.layer_scale2.lambda1"]
        converted[f"{target}.mlp.w12.weight"] = hf_state[f"{source}.mlp.weights_in.weight"]
        converted[f"{target}.mlp.w12.bias"] = hf_state[f"{source}.mlp.weights_in.bias"]
        converted[f"{target}.mlp.w3.weight"] = hf_state[f"{source}.mlp.weights_out.weight"]
        converted[f"{target}.mlp.w3.bias"] = hf_state[f"{source}.mlp.weights_out.bias"]
    return converted

torch.manual_seed(SEED)
np.random.seed(SEED)
hf_state = load_file(str(WEIGHTS_PATH), device="cpu")
MODEL = build_model(**MODEL_CONFIG)
MODEL.load_state_dict(convert_hf_state_dict(hf_state), strict=True)
del hf_state
gc.collect()
MODEL = MODEL.to(DEVICE).eval()
IMAGE = Image.open(IMAGE_PATH).convert("RGB")
PREPROCESS = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])
INPUT = PREPROCESS(IMAGE).unsqueeze(0).to(DEVICE)

def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)

print("模型 ID:", MODEL_ID)
print("PyTorch:", torch.__version__)
print("设备:", torch.cuda.get_device_name(DEVICE))
print("参数量:", sum(parameter.numel() for parameter in MODEL.parameters()))
print("输入:", IMAGE_PATH, "原始尺寸=", IMAGE.size, "模型张量=", tuple(INPUT.shape))
print("权重:", WEIGHTS_PATH)
print("外部结果目录:", RESULT_DIR)

模型 ID: histai/hibou-L
PyTorch: 2.6.0+metax3.8.2.2
设备: MetaX C500
参数量: 303659264
输入: /tmp/hibou_src/images/sample.png 原始尺寸= (1360, 1360) 模型张量= (1, 3, 224, 224)
权重: /tmp/hibou-L/model.safetensors
外部结果目录: /tmp/hibou-L-results


## 2. 沐曦 GPU 运行效果展示

### 2.1 CLS embedding

对应上游示例的 `output = model(data)`；结果文件只写入外部 `HIBOU_RESULT_DIR`，Notebook 同时保存形状、设备和实际耗时。

In [2]:
sync_device()
started = time.perf_counter()
with torch.no_grad():
    cls_embedding = MODEL(INPUT)
sync_device()
elapsed = time.perf_counter() - started
cls_cpu = cls_embedding.detach().float().cpu()
torch.save(cls_cpu, RESULT_DIR / "hibou_l_cls_embedding.pt")
summary = {
    "task": "model(image)",
    "shape": list(cls_cpu.shape),
    "dtype": str(cls_cpu.dtype),
    "device": str(DEVICE),
    "elapsed_seconds": elapsed,
    "l2_norm": float(torch.linalg.vector_norm(cls_cpu).item()),
    "mean": float(cls_cpu.mean().item()),
    "std": float(cls_cpu.std().item()),
}
(RESULT_DIR / "hibou_l_cls_embedding.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(summary)

{'task': 'model(image)', 'shape': [1, 1024], 'dtype': 'torch.float32', 'device': 'cuda:0', 'elapsed_seconds': 3.9703762610442936, 'l2_norm': 33.56162643432617, 'mean': 0.02015286311507225, 'std': 1.0491195917129517}


### 2.2 Patch tokens

对应上游 `forward_features` 路径。224×224 输入、14×14 patch 产生 16×16（共 256 个）patch tokens。

In [3]:
sync_device()
started = time.perf_counter()
with torch.no_grad():
    feature_dict = MODEL.forward_features(INPUT)
sync_device()
elapsed = time.perf_counter() - started
patch_tokens = feature_dict["x_norm_patchtokens"].detach().float().cpu()
torch.save(patch_tokens, RESULT_DIR / "hibou_l_patch_tokens.pt")
token_grid = int(patch_tokens.shape[1] ** 0.5)
if token_grid * token_grid != patch_tokens.shape[1]:
    raise ValueError(f"patch token 数不是平方数：{patch_tokens.shape[1]}")
token_norms = torch.linalg.vector_norm(patch_tokens[0], dim=-1)
summary = {
    "task": "model.forward_features(image)",
    "shape": list(patch_tokens.shape),
    "grid": [token_grid, token_grid],
    "device": str(DEVICE),
    "elapsed_seconds": elapsed,
    "token_l2_range": [float(token_norms.min().item()), float(token_norms.max().item())],
}
(RESULT_DIR / "hibou_l_patch_tokens.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(summary)

{'task': 'model.forward_features(image)', 'shape': [1, 256, 1024], 'grid': [16, 16], 'device': 'cuda:0', 'elapsed_seconds': 0.012227226048707962, 'token_l2_range': [33.68267059326172, 48.63798141479492]}


### 2.3 24 层中间表示

对应上游 `forward_features(image, return_intermediate=True)`，用于检查下游任务可使用的层级表示；保存最后一层张量到外部结果目录。

In [4]:
sync_device()
started = time.perf_counter()
with torch.no_grad():
    intermediate_dict = MODEL.forward_features(INPUT, return_intermediate=True)
sync_device()
elapsed = time.perf_counter() - started
intermediate = intermediate_dict["intermediate"]
last_intermediate = intermediate[-1].detach().float().cpu()
torch.save(last_intermediate, RESULT_DIR / "hibou_l_intermediate_last.pt")
summary = {
    "task": "model.forward_features(image, return_intermediate=True)",
    "layers": len(intermediate),
    "last_shape": list(last_intermediate.shape),
    "device": str(DEVICE),
    "elapsed_seconds": elapsed,
}
(RESULT_DIR / "hibou_l_intermediate.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(summary)

{'task': 'model.forward_features(image, return_intermediate=True)', 'layers': 24, 'last_shape': [1, 261, 1024], 'device': 'cuda:0', 'elapsed_seconds': 0.012623381335288286}


---
本文档仅提供相关软件的配置与使用说明，不包含亦不分发前述软件的源代码或目标代码，且不涉及对其源代码的任何修改。您按照本文档配置、部署或使用相关软件时，应遵守适用许可证规定的条款及条件。相关软件的源代码、许可证全文、版权与归属声明及其他项目文档，请以上游官方网站或原始发布页面为准。

Copyright (c) 2026 MetaX Integrated Circuits (Shanghai) Co., Ltd. All rights reserved.